# Figures: one language vector for many models

The figures of the article, in order. Figures 2–8 read results from `continual_pretrain/eval_results/`
(written by `scripts/language_vector/01-04`); figure 9 computes the weight comparison here. Each saves a PNG to `images/language_vector/`.

Figure 1 (the overview diagram) is not a plot: it is `overview.html`, saved as `01_overview.png` with a screenshot.

In [ ]:
import glob
import json
import statistics
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from matplotlib.lines import Line2D

from deltas import BASE, compute_deltas, load_state_dict, match_key, resolve
from euroeval_table import load as load_euroeval

RES = Path("../../eval_results")
BENCH, EMBED, RERANK = RES / "chat_vector_merge/bench", RES / "embed_merge", RES / "rerank_merge"
OUT = Path("../../../images/language_vector")
OUT.mkdir(parents=True, exist_ok=True)

BLUE, ORANGE, AQUA, YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
BLUE_LIGHT, ORANGE_LIGHT = "#a9c8ee", "#f6bba3"
INK, INK2, MUTED, GRID, AXIS, NOISE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#f0efec"
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 10, "axes.edgecolor": AXIS, "axes.labelcolor": INK2,
    "xtick.color": INK2, "ytick.color": INK2, "axes.spines.top": False, "axes.spines.right": False,
    "axes.titleweight": "bold", "axes.titlesize": 11, "axes.titlecolor": INK,
})

MTEB_NAMES = {
    "RomanianReviewsSentiment.v2": "Reviews sentiment", "RomanianSentimentClassification.v2": "Sentiment classification",
    "MassiveIntentClassification": "MASSIVE intent", "SIB200Classification.v3": "SIB200 topic classification",
    "SIB200ClusteringS2S": "SIB200 topic clustering", "WikipediaRerankingMultilingual": "Wikipedia rerank",
    "WikipediaRetrievalMultilingual": "Wikipedia retrieval", "XQuADRetrieval": "XQuAD retrieval",
    "BelebeleRetrieval": "Belebele retrieval", "FloresBitextMining": "FLORES bitext mining",
    "Tatoeba": "Tatoeba ro–en matching", "Moroco.v2": "MOROCO dialect", "WebFAQRetrieval": "WebFAQ retrieval",
    "RonSTS": "RonSTS similarity"}


def lm(model, task):
    """(score, standard error) of a 0-shot lm-eval run; accuracies in %."""
    r = json.load(open(sorted(glob.glob(str(BENCH / model / task / "fs0/*/results_*.json")))[-1]))["results"][task]
    metric = {"ro_wiki": "word_perplexity", "wikitext": "word_perplexity",
              "_ro_arc_challenge": "acc_norm", "_ro_hellaswag": "acc_norm"}.get(task, "acc")
    v, se = r[f"{metric},none"], r.get(f"{metric}_stderr,none")
    se = se if isinstance(se, float) else None
    return (v, se) if metric == "word_perplexity" else (100 * v, se and 100 * se)


def mteb(d):
    """MTEB main score per task, x100."""
    s = json.load(open(d / "summary.json"))["scores"]
    return {k: 100 * (v["main_score"] if isinstance(v, dict) else v) for k, v in s.items()}


def mteb_mean(d):
    s = mteb(d)
    return sum(s.values()) / len(s)


def euroeval(model, root=RES / "euroeval"):
    """dataset -> (score, 95% CI half-width)."""
    return {ds: (v, ci) for ds, (_, v, ci) in load_euroeval(root / model / "euroeval_benchmark_results.jsonl").items()}


def style(ax, axis="y"):
    ax.grid(axis=axis, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(length=0)
    if axis == "x":
        ax.spines["left"].set_visible(False)


def legend(fig, items, y, ncol=None):
    handles = [plt.Rectangle((0, 0), 1, 1, color=c, label=n) for n, c in items]
    fig.legend(handles=handles, loc="upper left", ncol=ncol or len(items), frameon=False, bbox_to_anchor=(0.005, y),
               fontsize=9.5, labelcolor=INK)


def title(fig, text, y=0.98):
    fig.suptitle(text, fontsize=13, fontweight="bold", color=INK, y=y)


def save(fig, name):
    fig.savefig(OUT / f"{name}.png", dpi=200, facecolor="white", bbox_inches="tight", pad_inches=0.25)
    plt.show()


def bar_label(ax, v, y, fmt="{:+.1f}"):
    """Value label just past the end of a horizontal bar."""
    ax.annotate(fmt.format(v).replace("+0.0", "0.0").replace("-0.0", "0.0"), (v, y), xytext=(4 if v >= 0 else -4, 0),
                textcoords="offset points", ha="left" if v >= 0 else "right", va="center", fontsize=8.5, color=INK2)

## 2. Romanian knowledge and understanding (OpenLLM-Ro, 0-shot)

In [ ]:
models = [("base", "Base", YELLOW), ("rollama", "Base + CPT", AQUA),
          ("instruct", "Instruct", ORANGE), ("lambda_0.5_novocab", "Instruct + vector", BLUE)]
tasks = [("_ro_arc_challenge", "RO ARC ↑", 25), ("_ro_hellaswag", "RO HellaSwag ↑", 25), ("_ro_mmlu", "RO MMLU ↑", 25),
         ("_ro_winogrande", "RO Winogrande ↑", 50), ("_ro_truthfulqa", "RO TruthfulQA ↑", None),
         ("ro_wiki", "RO Wikipedia perplexity ↓", None)]

fig, axes = plt.subplots(2, 3, figsize=(10.5, 6.4))
title(fig, "Romanian knowledge and understanding", y=0.995)
for ax, (task, name, chance) in zip(axes.flat, tasks):
    vals = [lm(m, task) for m, *_ in models]
    for i, ((_, _, c), (v, se)) in enumerate(zip(models, vals)):
        if se:
            ax.errorbar(i, v, yerr=se, color=c, linewidth=2, zorder=2)
        ax.scatter(i, v, s=90, color=c, edgecolor="white", linewidth=2, zorder=3)
        ax.annotate(f"{v:.1f}", (i, v), xytext=(9, 0), textcoords="offset points", va="center", fontsize=9, color=INK2)
    lo, hi = min(v - (se or 0) for v, se in vals), max(v + (se or 0) for v, se in vals)
    if chance and chance >= lo - 4:
        ax.axhline(chance, color=MUTED, linestyle=(0, (4, 3)), linewidth=1, zorder=1)
        ax.annotate("chance", (-0.45, chance), va="bottom", fontsize=8, color=MUTED)
        lo = min(lo, chance)
    pad = (hi - lo) * 0.18
    ax.set(ylim=(lo - pad, hi + pad), xlim=(-0.5, 3.6), xticks=[])
    ax.set_title(name, loc="left")
    style(ax)
handles = [Line2D([], [], marker="o", linestyle="", markersize=9, color=c, label=n) for _, n, c in models]
fig.legend(handles=handles, loc="upper center", ncol=4, frameon=False, bbox_to_anchor=(0.5, 0.965), labelcolor=INK)
fig.tight_layout(rect=(0, 0.03, 1, 0.93))
save(fig, "02_knowledge")

## 3. Romanian assistant tasks (EuroEval, chat template)

In [ ]:
EUROEVAL_RO = [("multi-ifeval-ro", "Instruction following (IFEval)"), ("flores-en-ro", "Translation en→ro, FLORES (chrF)"),
               ("wmt24pp-en-ro", "Translation en→ro, WMT24++ (chrF)"), ("ro-sent", "Sentiment (MCC)"),
               ("ronec", "Named entities (F1)"), ("sumo-ro", "Summarization (chrF)"),
               ("multi-wiki-qa-ro", "Question answering (F1)"), ("ragtruth-ro", "Hallucination rate ↓")]
a, b = euroeval("instruct"), euroeval("vector")

fig, ax = plt.subplots(figsize=(9, 5.6))
title(fig, "Romanian assistant tasks")
h = 0.36
y = range(len(EUROEVAL_RO) - 1, -1, -1)
for yi, (ds, _) in zip(y, EUROEVAL_RO):
    (va, ea), (vb, eb) = a[ds], b[ds]
    is_clear = abs(vb - va) > (ea ** 2 + eb ** 2) ** 0.5  # difference outside the combined 95% CI
    for off, v, (c, cl) in ((h / 2 + 0.02, va, (ORANGE, ORANGE_LIGHT)), (-h / 2 - 0.02, vb, (BLUE, BLUE_LIGHT))):
        ax.barh(yi + off, v, height=h, color=c if is_clear else cl, zorder=2)
        bar_label(ax, v, yi + off, "{:.1f}")
ax.set_yticks(list(y), [lab for _, lab in EUROEVAL_RO], fontsize=9.5)
ax.set(ylim=(-0.6, len(EUROEVAL_RO) - 0.4), xlim=(0, 85), xlabel="Score (0–100)")
style(ax, "x")
legend(fig, [("Instruct", ORANGE), ("Instruct + vector", BLUE), ("lighter = within noise", GRID)], 0.935)
fig.subplots_adjust(left=0.31, right=0.97, top=0.86, bottom=0.16)
save(fig, "03_assistant_tasks")

## 4. Romanian embedding and reranking (MTEB)

Embedding-model tasks that moved by at least 1 point get their own bar; the rest are folded into one range.
Only Tatoeba and MASSIVE are clearly above noise: MASSIVE over MTEB's 10 paired runs (+4.0 ± 0.8),
Tatoeba against ~±2.3 from its 1000 pairs. Clustering and review sentiment: the paired 95% CI includes 0.
RonSTS and the reranker have no noise estimate.

In [ ]:
a, b = mteb(EMBED / "nv-embed"), mteb(EMBED / "nv-embed_l0.5_novocab")
d = {k: b[k] - a[k] for k in a}
big = sorted((k for k in d if d[k] >= 1), key=lambda k: -d[k])
rest = [k for k in d if k not in big]
ra, rb = mteb(RERANK / "nv-rerank"), mteb(RERANK / "nv-rerank_l0.5_novocab")
rma, rmb = sum(ra.values()) / len(ra), sum(rb.values()) / len(rb)
clear = {"Tatoeba", "MassiveIntentClassification"}

rows = [("Embedding model", None, "", "head")]
rows += [(MTEB_NAMES[k], d[k], f"{a[k]:.1f} → {b[k]:.1f}", "clear" if k in clear else "noise") for k in big]
rows += [(f"{len(rest)} other tasks*", (min(d[k] for k in rest), max(d[k] for k in rest)), "each changed by less than 0.3", "range")]
rows += [(f"Reranker (average of {len(ra)} tasks)**", rmb - rma, f"{rma:.1f} → {rmb:.1f}", "mean")]

fig, ax = plt.subplots(figsize=(9, 5.1))
title(fig, "Romanian embedding and reranking")
ys = [-(i + 1) - (0.6 if i == len(rows) - 1 else 0) for i in range(len(rows))]  # extra gap before the reranker
ax.axvline(0, color=AXIS, linewidth=1, zorder=1)
for yi, (lab, v, note, kind) in zip(ys, rows):
    if kind == "range":
        ax.barh(yi, v[1] - v[0], left=v[0], height=0.6, color=GRID, zorder=2)
        x, txt = v[1], note
    elif kind != "head":
        ax.barh(yi, v, height=0.6, color=BLUE if kind == "clear" else BLUE_LIGHT, zorder=2)
        x, txt = v, f"{v:+.1f}   {note}"
    if kind != "head":
        ax.annotate(txt, (max(x, 0), yi), xytext=(5, 0), textcoords="offset points", va="center", fontsize=8.5, color=INK2)
ax.set_yticks(ys, [r[0] for r in rows], fontsize=9.5)
for t, r in zip(ax.get_yticklabels(), rows):
    if r[3] in ("head", "mean"):
        t.set_fontweight("bold")
        t.set_color(INK)
ax.set(ylim=(ys[-1] - 0.7, ys[0] + 0.7), xlim=(-0.5, 6.5), xlabel="Change from adding the language vector (MTEB score × 100)")
style(ax, "x")
legend(fig, [("Clear gain", BLUE), ("Within noise, or noise not measured", BLUE_LIGHT)], 0.935)
others = ", ".join(MTEB_NAMES[k] for k in sorted(rest, key=lambda k: -a[k]))
reranker_tasks = ", ".join(MTEB_NAMES[k] for k in sorted(ra, key=lambda k: ra[k] - rb[k]))
fig.text(0.012, 0.015, textwrap.fill(f"* {others}.", 135) + "\n" + textwrap.fill(
    f"** {reranker_tasks}; the reranker reorders the embedding model's top 20 candidates.", 135), fontsize=8.5, color=MUTED)
fig.subplots_adjust(left=0.33, right=0.97, top=0.86, bottom=0.26)
save(fig, "04_embedding_reranking")

## 5. Forgetting: chat model, Romanian vs English (EuroEval)

Lighter bars: the change is smaller than the combined 95% confidence intervals.

In [ ]:
COST_XLIM, COST_XLABEL = (-4.5, 7.5), "Change from adding the language vector (points; positive = better)"
COST_LEGEND = [("Romanian", BLUE), ("English", ORANGE), ("lighter = within noise", GRID)]
ro_a, ro_b = euroeval("instruct"), euroeval("vector")
en_a, en_b = euroeval("instruct", RES / "euroeval_en"), euroeval("vector", RES / "euroeval_en")
pairs = [("Instruction following", "multi-ifeval-ro", "ifeval"), ("Sentiment", "ro-sent", "sst5"),
         ("Named entities", "ronec", "conll-en"), ("Summarization", "sumo-ro", "cnn-dailymail"),
         ("Question answering", "multi-wiki-qa-ro", "squad"), ("Hallucination (lower rate = better)", "ragtruth-ro", "ragtruth-en")]


def change(a, b, ds):
    """Improvement (hallucination rate flipped) and whether it is outside the combined CI."""
    (va, ea), (vb, eb) = a[ds], b[ds]
    dlt = (vb - va) * (-1 if ds.startswith("ragtruth") else 1)
    return dlt, abs(dlt) > (ea ** 2 + eb ** 2) ** 0.5


fig, ax = plt.subplots(figsize=(9, 4.9))
title(fig, "Chat model: Romanian vs English")
y = range(len(pairs) - 1, -1, -1)
ax.axvline(0, color=AXIS, linewidth=1, zorder=1)
for yi, (_, ro, en) in zip(y, pairs):
    for off, (dlt, is_clear), (c, cl) in ((h / 2 + 0.02, change(ro_a, ro_b, ro), (BLUE, BLUE_LIGHT)),
                                          (-h / 2 - 0.02, change(en_a, en_b, en), (ORANGE, ORANGE_LIGHT))):
        ax.barh(yi + off, dlt, height=h, color=c if is_clear else cl, zorder=2)
        bar_label(ax, dlt, yi + off)
ax.set_yticks(list(y), [p[0] for p in pairs], fontsize=9.5)
ax.set(ylim=(-0.6, len(pairs) - 0.4), xlim=COST_XLIM, xlabel=COST_XLABEL)
style(ax, "x")
legend(fig, COST_LEGEND, 0.935)
fig.subplots_adjust(left=0.3, right=0.97, top=0.85, bottom=0.2)
save(fig, "05_forgetting_chat")

## 6. Forgetting: embedding model and reranker, Romanian vs English subsets of the same MTEB datasets

A change counts as clear when MTEB's repeated runs (classification: 10 training samples, clustering: 10 restarts)
give a paired 95% CI that excludes 0. Tasks without repeated runs count as clear when they move by at least 1 point.

In [ ]:
def paired_runs_clear(root, task):
    """True/False from the paired t-test over MTEB's repeated runs; None when the task has none."""
    def runs(model):
        f = glob.glob(str(root / model / f"cache/results/*/*/{task}.json"))
        if not f:
            return None
        s = [x for split in json.load(open(f[0]))["scores"].values() for x in split][0]
        if "scores_per_experiment" in s:
            return [r["accuracy"] for r in s["scores_per_experiment"]]
        return next(iter(s["v_measures"].values())) if "v_measures" in s else None
    a, b = runs("nv-embed"), runs("nv-embed_l0.5_novocab")
    if not a or not b:
        return None
    diff = [y - x for x, y in zip(a, b)]
    sd = statistics.stdev(diff)
    return None if sd == 0 else abs(statistics.mean(diff)) > 2.26 * sd / len(diff) ** 0.5


def mteb_change(root, orig, new, keys):
    a, b = mteb(root / orig), mteb(root / new)
    return {k: b[k] - a[k] for k in keys}


emb_keys = ["MassiveIntentClassification", "SIB200Classification.v3", "SIB200ClusteringS2S", "WikipediaRerankingMultilingual",
            "BelebeleRetrieval", "XQuADRetrieval", "WikipediaRetrievalMultilingual"]
rr_keys = ["WikipediaRerankingMultilingual", "WikipediaRetrievalMultilingual", "XQuADRetrieval"]
panels = [("Embedding model", EMBED, "nv-embed", emb_keys), ("Reranker", RERANK, "nv-rerank", rr_keys)]

fig, axes = plt.subplots(2, 1, figsize=(9, 0.62 * (len(emb_keys) + len(rr_keys)) + 2.2), sharex=True,
                         gridspec_kw={"height_ratios": [len(emb_keys), len(rr_keys)], "hspace": 0.3})
title(fig, "Embedding model and reranker: Romanian vs English", y=0.995)
for ax, (name, root, orig, keys) in zip(axes, panels):
    ro = mteb_change(root, orig, f"{orig}_l0.5_novocab", keys)
    en = mteb_change(root / "en_paired", orig, f"{orig}_l0.5_novocab", keys)
    keys = sorted(keys, key=lambda k: -ro[k])
    y = range(len(keys) - 1, -1, -1)
    ax.axvline(0, color=AXIS, linewidth=1, zorder=1)
    for yi, k in zip(y, keys):
        for off, dlt, sub, (c, cl) in ((h / 2 + 0.02, ro[k], root, (BLUE, BLUE_LIGHT)),
                                       (-h / 2 - 0.02, en[k], root / "en_paired", (ORANGE, ORANGE_LIGHT))):
            is_clear = paired_runs_clear(sub, k) if root == EMBED else None
            is_clear = abs(dlt) >= 1 if is_clear is None else is_clear
            ax.barh(yi + off, dlt, height=h, color=c if is_clear else cl, zorder=2)
            bar_label(ax, dlt, yi + off)
    ax.set_yticks(list(y), [MTEB_NAMES[k] for k in keys], fontsize=9.5)
    ax.set_ylim(-0.6, len(keys) - 0.4)
    ax.set_title(name, loc="left")
    style(ax, "x")
axes[-1].set(xlim=COST_XLIM, xlabel=COST_XLABEL)
legend(fig, COST_LEGEND, 0.95)
fig.subplots_adjust(left=0.27, right=0.97, top=0.86, bottom=0.1)
save(fig, "06_forgetting_encoders")

## 7. How much of the vector to add (λ sweep, full vector including the vocabulary part)

In [ ]:
lambdas = [(0, "instruct"), (0.25, "lambda_0.25"), (0.5, "lambda_0.5"), (0.75, "lambda_0.75"), (1, "lambda_1")]
panels = [(name, [(lam, *lm(m, task)) for lam, m in lambdas]) for task, name in (
    ("_ro_hellaswag", "Chat · RO HellaSwag ↑"), ("_ro_mmlu", "Chat · RO MMLU ↑"),
    ("ro_wiki", "Chat · RO Wikipedia perplexity ↓"), ("wikitext", "Chat · EN Wikipedia perplexity ↓"))]
for name, root, orig in (("Embedder · RO MTEB mean ↑", EMBED, "nv-embed"), ("Reranker · RO mean ↑", RERANK, "nv-rerank")):
    panels.append((name, [(0, mteb_mean(root / orig), None)] +
                   [(lam, mteb_mean(root / f"{orig}_l{lam:g}"), None) for lam in (0.25, 0.5, 1)]))

fig, axes = plt.subplots(2, 3, figsize=(10.5, 6))
title(fig, "How much of the vector to add", y=0.99)
for ax, (name, pts) in zip(axes.flat, panels):
    ax.axvspan(0.44, 0.56, color=NOISE, zorder=0)
    ax.plot([p[0] for p in pts], [p[1] for p in pts], color=BLUE, linewidth=2, zorder=2)
    for x, v, e in pts:
        if e:
            ax.errorbar(x, v, yerr=e, color=BLUE, linewidth=1.2, zorder=2)
        ax.scatter(x, v, s=55, color=BLUE, edgecolor="white", linewidth=1.5, zorder=3)
    fmt = ".2f" if name.startswith(("Embedder", "Reranker")) else ".1f"
    for x, v, _ in (pts[0], next(p for p in pts if p[0] == 0.5), pts[-1]):  # label λ = 0, 0.5 and 1
        below = x == 0 and pts[1][1] > v                                      # rising line: label under the point
        left = x == 0.5 and [p[1] for p in pts if p[0] > x][0] > v            # still rising after 0.5: label up-left
        ax.annotate(f"{v:{fmt}}", (x, v), xytext=(-6 if left else 6, -5 if below else 5), textcoords="offset points",
                    ha="right" if left else "left", va="top" if below else "bottom", fontsize=8.5, color=INK2)
    lo, hi = min(v - (e or 0) for _, v, e in pts), max(v + (e or 0) for _, v, e in pts)
    pad = (hi - lo) * 0.22
    ax.set(ylim=(lo - pad, hi + pad * 1.6), xlim=(-0.08, 1.2))
    ax.set_xticks([0, 0.25, 0.5, 0.75, 1], ["0", ".25", ".5", ".75", "1"])
    ax.set_title(name, loc="left", fontsize=10)
    style(ax)
for ax in axes[-1]:
    ax.set_xlabel("λ (how much of the vector is added)")
fig.tight_layout(rect=(0, 0.035, 1, 0.95))
save(fig, "07_lambda")

## 8. Chat model with or without the vocabulary part (λ = 0.5)

"With vocabulary" keeps Llama-3.2-1B's shared input/output matrix and gives it the lm_head delta.

In [ ]:
versions = ["instruct", "lambda_0.5_tied-lm_head", "lambda_0.5_novocab"]
fig, ax = plt.subplots(figsize=(7.2, 4.4))
title(fig, "Chat model: with or without the vocabulary part")
for (task, lab), c in zip((("_ro_hellaswag", "RO HellaSwag ↑"), ("_ro_arc_challenge", "RO ARC ↑"),
                           ("ro_wiki", "RO perplexity ↓")), (BLUE, ORANGE, AQUA)):
    ys = [lm(m, task)[0] for m in versions]
    ax.plot(range(3), ys, color=c, linewidth=2.2, zorder=2)
    ax.scatter(range(3), ys, s=60, color=c, edgecolor="white", linewidth=1.5, zorder=3)
    for x, v in enumerate(ys):
        ax.annotate(f"{v:.1f}", (x, v), xytext=(0, 8), textcoords="offset points", ha="center", fontsize=8.5, color=INK2)
    ax.annotate(lab, (2, ys[-1]), xytext=(14, 0), textcoords="offset points", va="center", fontsize=9.5,
                color=c, fontweight="bold")
ax.set_xticks(range(3), ["Original", "With\nvocabulary", "Transformer\nlayers only"], fontsize=9.5)
ax.set(xlim=(-0.3, 3.1), ylim=(20, 44))
style(ax)
fig.subplots_adjust(left=0.08, right=0.98, top=0.88, bottom=0.15)
save(fig, "08_vocabulary")

## 9. Size and direction of the weight changes

For the 112 transformer matrices (16 layers × 7 projections), with W the base weights, ΔW the language vector
and D = W_model − W the change each model's own fine-tuning made:
- size: ||ΔW|| / ||W|| and ||D|| / ||W||
- direction: cosine of the flattened matrices; cos(D_embed, D_rerank) as a related pair; 1/√(#entries) as the level two random directions reach

The next cell loads four 1B models and compares them on the GPU (a few minutes). It saves the means to `eval_results/weight_geometry/weights.json`.

In [ ]:
MODELS = {"instruct": "unsloth/Llama-3.2-1B-Instruct",
          "embed": "nvidia/llama-3.2-nv-embedqa-1b-v2",
          "rerank": "nvidia/llama-3.2-nv-rerankqa-1b-v2"}


def cos(a, b):
    return ((a * b).sum() / (a.norm() * b.norm())).item()


base = load_state_dict(resolve(BASE))
vector = {k: v for k, v in compute_deltas(base=base).items() if ".layers." in k}
models = {m: load_state_dict(resolve(mid)) for m, mid in MODELS.items()}

rows = []
for name, dW in vector.items():
    W, dW = base[name].float().cuda(), dW.cuda()
    D = {m: sd[match_key(name, sd)].float().cuda() - W for m, sd in models.items()}
    rows.append({"vector_size": (dW.norm() / W.norm()).item(),
                 **{f"{m}_size": (D[m].norm() / W.norm()).item() for m in D},
                 **{f"vector_vs_{m}": cos(dW, D[m]) for m in D},
                 "embed_vs_rerank": cos(D["embed"], D["rerank"]),
                 "random": W.numel() ** -0.5})

w = {k: sum(r[k] for r in rows) / len(rows) for k in rows[0]}
(RES / "weight_geometry").mkdir(exist_ok=True)
(RES / "weight_geometry/weights.json").write_text(json.dumps({"matrices": len(rows), **w}, indent=2))
del base, vector, models
w

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6.6), gridspec_kw={"height_ratios": [5, 4], "hspace": 0.62})
title(fig, "Size and direction of the weight changes", y=0.985)

rows = [("Romanian vector (λ = 1)", w["vector_size"], BLUE), ("Romanian vector (λ = 0.5)", w["vector_size"] / 2, BLUE),
        ("Instruct's own fine-tuning", w["instruct_size"], MUTED), ("nv-embed's own fine-tuning", w["embed_size"], MUTED),
        ("nv-rerank's own fine-tuning", w["rerank_size"], MUTED)]
y = range(len(rows) - 1, -1, -1)
for yi, (_, v, c) in zip(y, rows):
    ax1.barh(yi, 100 * v, height=0.6, color=c, zorder=2)
    ax1.annotate(f"{100 * v:.1f}%", (100 * v, yi), xytext=(5, 0), textcoords="offset points", va="center", fontsize=10, color=INK2)
ax1.set_yticks(list(y), [r[0] for r in rows], fontsize=10.5)
ax1.set(xlim=(0, 32), xlabel="Size of the weight change, relative to the weights (%)")
ax1.set_title("How far each change moves Llama-3.2-1B", loc="left")
style(ax1, "x")

pts = [("Vector vs nv-rerank's fine-tuning", w["vector_vs_rerank"], BLUE),
       ("Vector vs nv-embed's fine-tuning", w["vector_vs_embed"], BLUE),
       ("Vector vs Instruct's fine-tuning", w["vector_vs_instruct"], BLUE),
       ("nv-embed vs nv-rerank (related)", w["embed_vs_rerank"], MUTED)]
y = range(len(pts) - 1, -1, -1)
ax2.axvline(w["random"], color=MUTED, linestyle=(0, (4, 3)), linewidth=1.2, zorder=1)
ax2.annotate("random chance", (w["random"], -0.55), xytext=(4, 0), textcoords="offset points", fontsize=9, color=MUTED, va="bottom")
for yi, (_, v, c) in zip(y, pts):
    ax2.hlines(yi, 1e-4, v, color=GRID, linewidth=1.5, zorder=1)
    ax2.scatter(v, yi, s=80, color=c, edgecolor="white", linewidth=1.5, zorder=3)
    ax2.annotate(f"{v:.4f}" if v < 0.01 else f"{v:.3f}", (v, yi), xytext=(8, 0), textcoords="offset points",
                 va="center", fontsize=10, color=INK2)
ax2.set_xscale("log")
ax2.set_yticks(list(y), [p[0] for p in pts], fontsize=10.5)
ax2.set(xlim=(1e-4, 0.2), ylim=(-0.6, len(pts) - 0.4), xlabel="Cosine similarity (log scale; 1 = same direction)")
ax2.set_title("How aligned the changes are", loc="left")
style(ax2, "x")
fig.subplots_adjust(left=0.36, right=0.95, top=0.9, bottom=0.09)
save(fig, "09_weights")